In [21]:
#imports
import pandas as pd


In [22]:
#Train dataset
train = pd.read_csv('train.csv')

#print(f'{train.describe()}\n')
#print(f'{train.info()}\n')
print(f'{train.isnull().sum()}\n')

robot_id              0
arena_type            0
weather               0
difficulty            0
laps                  0
avg_speed_mps         0
distance_m            0
battery_used_pct      0
collisions            0
unique_cells          0
items_collected       0
pause_count           0
turn_rate             0
scan_accuracy         0
return_time_s         0
efficiency_score      0
coverage_ratio        0
scan_turn_sync        0
pickup_efficiency     0
detour_index          0
safety_margin         0
patrol_consistency    0
speed_burst           0
risk_index            0
strategy_label        0
dtype: int64



In [23]:
#Pandas Tasks

T1 = len(pd.unique(train['arena_type']))

T2 = train['avg_speed_mps'].max()

T3 = train['arena_type'].mode()[0]

T4 = train['items_collected'].max()

T14 = pd.DataFrame({
    'robot_id':'GLOBAL',
    'subtaskID':[1,2,3,4],
    'answer':[T1,T2,T3,T4]
})

In [24]:
from sklearn.preprocessing import OneHotEncoder,StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import f1_score

In [25]:
X = train.drop(columns=(['robot_id','strategy_label']))
y = train['strategy_label']

cat_cols = ['arena_type','weather']
num_cols = [c for c in X.columns if c not in cat_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ('num',StandardScaler(),num_cols),
        ('cat',OneHotEncoder(),cat_cols)
    ]
)

X_processed = preprocessor.fit_transform(X)

label = LabelEncoder()

y = label.fit_transform(y)

In [26]:
X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, random_state=42)

logReg = LogisticRegressionCV()

logReg.fit(X_train,y_train)

y_pred = logReg.predict(X_test)

print(f'Macro F1 Score: {f1_score(y_true=y_test,y_pred=y_pred,average='macro')}\n')

/home/moldo/Projects/MLCompete/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/home/moldo/Projects/MLCompete/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of LogisticRegressionCV for more details.
  warnings.warn(


Macro F1 Score: 0.9247426030721264



In [27]:
test = pd.read_csv('test.csv')

X_raw = test.drop(columns=('robot_id'))

X_encoded = preprocessor.transform(X_raw)

predictions = logReg.predict(X_encoded)

predictions = label.inverse_transform(predictions)

In [28]:
T5 = pd.DataFrame({
    'robot_id':test['robot_id'],
    'subtaskID':5,
    'answer':predictions
})

submission = pd.concat((T14,T5))
submission.columns = ['robot_id','subtaskID','answer']
submission.to_csv('submission.csv', index=False)